**WARNING** We don't like the interpretation of the notebook, possibly due to following the tutorial incorrectly. We keep it but won't use it for downstream analyses.

In [ ]:
import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import os
import shutil
import triku as tk
import matplotlib.pyplot as plt
import matplotlib as mpl
import subprocess
from scipy.sparse import csr_matrix
from IPython.display import display, HTML
import mygene as mg
import seaborn as sns
from tqdm import tqdm
import phate
# from tqdm.notebook import tqdm

from bokeh.io import show, output_notebook, reset_output

from scipy.sparse import csr_matrix, csc_matrix

reset_output()
output_notebook()

In [ ]:
from pyfuncs.general import preprocessing_adata_sub, plot_volcano


In [ ]:
#!pip install pegasuspy

# Waddington-OT

In [ ]:
os.makedirs('data/oprescu-WOT', exist_ok=True)

In [ ]:
import wot
import pegasus as pg
import pegasusio as io
from matplotlib.colors import ListedColormap, BoundaryNorm

## Compute FLE coordinates

In [ ]:
adata_var = wot.io.read_dataset('data/oprescu-WOT/adata_oprescu_onlyFAP_clean.h5ad')
adata_var

In [ ]:
preprocessing_adata_sub(adata_var, integrate=False)


In [ ]:
mmdata = io.MultimodalData(adata_var)

In [ ]:
pg.pca(mmdata, features=None)
pg.neighbors(mmdata, rep='pca') # set rep='pca_harmony' for batch correction

In [ ]:
pg.diffmap(mmdata, rep='pca') # set rep='pca_harmony' for batch correction

In [ ]:
pg.fle(mmdata)

In [ ]:
cats = ['X0.5.DPI', 'X2.DPI', 'X3.5.DPI', 'X5.DPI', 'X10.DPI', 'X21.DPI', 'Noninjured']
colors7 = ["#0e0b2b", "#400f74", "#762181", "#ad347c", "#e34e65", "#fb8761", "#fec68a"]

coords = mmdata.obsm["X_fle"]

# batch -> categorical con tu orden
mmdata.obs["batch"] = pd.Categorical(mmdata.obs["batch"].astype(str), categories=cats, ordered=True)
codes = mmdata.obs["batch"].cat.codes  # 0..6, y -1 si hay valores fuera de cats o NaN

cmap = ListedColormap(colors7)
norm = BoundaryNorm(np.arange(len(cats) + 1) - 0.5, len(cats))

plt.figure(figsize=(10, 10))
plt.axis("off")

# pinta los "-1" (desconocidos) en gris claro
mask_known = codes != -1
plt.scatter(coords[~mask_known, 0], coords[~mask_known, 1], c="#d9d9d9", s=6, linewidths=0)
sc = plt.scatter(coords[mask_known, 0], coords[mask_known, 1], c=codes[mask_known], s=6, cmap=cmap, norm=norm, linewidths=0)

cbar = plt.colorbar(sc, ticks=np.arange(len(cats)))
cbar.ax.set_yticklabels(cats)
cbar.ax.set_title("Day")
plt.tight_layout()



In [ ]:
# guarda coords en el adata para que todo quede autocontenido
adata_var.obsm["X_fle"] = mmdata.obsm["X_fle"].copy()

# dataframe de coords (index = cell ids)
fle_df = pd.DataFrame(
    adata_var.obsm["X_fle"],
    index=adata_var.obs_names,
    columns=["fle1", "fle2"],
)
fle_df.head()

In [ ]:
cats = ['X0.5.DPI', 'X2.DPI', 'X3.5.DPI', 'X5.DPI', 'X10.DPI', 'X21.DPI', 'Noninjured']

day_map = {
    "Noninjured": 0.0,   # baseline
    "X0.5.DPI": 0.5,
    "X2.DPI": 2.0,
    "X3.5.DPI": 3.5,
    "X5.DPI": 5.0,
    "X10.DPI": 10.0,
    "X21.DPI": 21.0,
}

adata_wot = adata_var.copy()
adata_wot.obs["batch"] = adata_wot.obs["batch"].astype(str)
adata_wot.obs["day"] = adata_wot.obs["batch"].map(day_map).astype(float)

# sanity check
adata_wot.obs[["batch", "day"]].drop_duplicates().sort_values("day")

In [ ]:
tmap_dir = "data/oprescu-WOT/tmaps"
os.makedirs(tmap_dir, exist_ok=True)

# prefijo de salida: generará ficheros tipo  oprescu_0.0_0.5.h5ad  etc.
tmap_prefix = os.path.join(tmap_dir)

ot_model = wot.ot.OTModel(
    adata_wot,
    day_field="day",
    epsilon=0.05,
    lambda1=2,
    lambda2=50,
    local_pca=50,
    growth_iters=3
)

ot_model.compute_all_transport_maps(tmap_out=tmap_prefix, overwrite=True, output_file_format="h5ad")

In [ ]:
tmap_model = wot.tmap.TransportMapModel.from_directory(tmap_dir)

# ejemplo: coupling largo 0.5 -> 10 (si no es adyacente, compone internamente)
gamma = tmap_model.get_coupling(10, 21)
gamma

In [ ]:
# Visualize how growth rates change with growth iterations
plt.scatter(gamma.obs['g2'],gamma.obs['g3'])
plt.xlabel("g0")
plt.ylabel("g1")
plt.title("Input vs Output Growth Rates")
plt.show()

In [ ]:
candidate_keys = ["merged_clusters", "leiden", "louvain", "clusters", "cell_type", "annotation"]
cluster_key = next((k for k in candidate_keys if k in adata_wot.obs.columns), None)
cluster_key

In [ ]:
terminal_day = 21.0

terminal = adata_wot.obs["day"] == terminal_day
labels = adata_wot.obs.loc[terminal, cluster_key].astype(str)

cell_sets = {
    f"{cluster_key}:{lab}": set(labels.index[labels == lab])
    for lab in sorted(labels.unique())
}

len(cell_sets), list(cell_sets)[:5]

In [ ]:
populations = tmap_model.population_from_cell_sets(cell_sets, at_time=terminal_day)
fate_ds = tmap_model.fates(populations)   # AnnData: filas=células (hasta terminal_day), cols=fates

fate_ds

In [ ]:
# unir coords al obs del fate_ds
fate_ds.obs = fate_ds.obs.join(fle_df, how="left")

# para cada célula: qué fate tiene mayor prob
fate_names = fate_ds.var_names
fate_max = pd.Series(fate_names[np.asarray(fate_ds.X).argmax(axis=1)], index=fate_ds.obs_names)

fate_ds.obs["fate_max"] = fate_max.values
fate_ds.obs.loc[fate_ds.obs_names, 'population'] = adata_var.obs.loc[fate_ds.obs_names, 'merged_clusters']
fate_ds.obs[["day", "population", "fate_max"]]

In [ ]:
fate_ds[fate_ds.obs['day'] == 0.5].obs

In [ ]:
import matplotlib.pyplot as plt

day_to_plot = 5
mask = fate_ds.obs["day"] == day_to_plot

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.scatter(fate_ds.obs.loc[mask, "fle1"], fate_ds.obs.loc[mask, "fle2"],
            c=pd.Categorical(fate_ds.obs.loc[mask, "fate_max"]).codes,
            s=6, linewidths=0)
plt.title(f"Most-likely fate @ day {day_to_plot}")
plt.tight_layout()

In [ ]:
traj_ds = tmap_model.trajectories(populations)
traj_ds.obs = traj_ds.obs.join(fle_df, how="left")
traj_ds

In [ ]:
traj_ds.var_names

In [ ]:
name = traj_ds.var_names[2]  # cambia a la trayectoria/fate que te interese

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.scatter(traj_ds.obs["fle1"], traj_ds.obs["fle2"], c="#f0f0f0", s=4, linewidths=0)
plt.scatter(traj_ds.obs["fle1"], traj_ds.obs["fle2"], c=np.asarray(traj_ds[:, name].X).ravel(),
            s=6, linewidths=0)
plt.colorbar().ax.set_title("trajectory")
plt.title(name)
plt.tight_layout()

In [ ]:
start_day = 0

start_labels = adata_wot.obs.loc[adata_wot.obs["day"] == start_day, cluster_key].astype(str)
start_cell_sets = {
    f"{cluster_key}:{lab}": set(start_labels.index[start_labels == lab])
    for lab in sorted(start_labels.unique())
}

start_pops = tmap_model.population_from_cell_sets(start_cell_sets, at_time=start_day)
end_pops   = populations  # ya eran los de terminal_day

tt = tmap_model.transition_table(start_pops, end_pops)
tt

In [ ]:
import pandas as pd

tt_df = pd.DataFrame(tt.X, index=tt.obs_names, columns=tt.var_names)
tt_df.iloc[:5, :5]